In [69]:
!pip install langchain
!pip install langchain_community
!pip install langchain_text_splitter
!pip install langchain_chroma
!pip install langchain_groq

ERROR: Could not find a version that satisfies the requirement langchain_text_splitter (from versions: none)
ERROR: No matching distribution found for langchain_text_splitter


In [70]:
import os
#from langchain.document_loaders import UnstructuredFileLoader #old import not working
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [ ]:
os.environ["GROQ_API_KEY"] = "Enter your actual GROQ API key here"

In [ ]:
!pip install -q langsmith

import os

from langsmith import Client

# GROQ API
os.environ["GROQ_API_KEY"] = "Enter your actual GROQ API key here"

# LANGSMITH
os.environ["LANGCHAIN_API_KEY"] = "Enter your actual LangChain API key here"

# ENABLE TRACING
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# PROJECT NAME
os.environ["LANGCHAIN_PROJECT"] = "RAG-UPGRADE-PROJECT"

print("LangSmith Tracing Enabled Successfully")

LangSmith Tracing Enabled Successfully


In [73]:
# Fetch the PDF from the URL
import requests
url = "https://dspmuranchi.ac.in/pdf/Blog/Python%20Built-In%20Functions.pdf"
response = requests.get(url)

In [74]:
# Save the PDF to a local file
with open("python_inbuildfunction.pdf", "wb") as f:
    f.write(response.content)

In [75]:
!pip install unstructured-inference
!pip install unstructured
!pip install "unstructured[pdf]"

In [76]:
# laoding the document
loader = UnstructuredFileLoader("python_inbuildfunction.pdf")

In [77]:
# Restart the session then this code will work

text_splitter = CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)

In [78]:
pip install unstructured-inference

Note: you may need to restart the kernel to use updated packages.


In [79]:
document = loader.load()
texts = text_splitter.split_documents(document)

No languages specified, defaulting to English.


In [80]:
type(texts)

list

In [81]:
len(texts)

7

In [82]:
texts[4]

Document(metadata={'source': 'python_inbuildfunction.pdf'}, page_content='frozenset() returns an immutable frozenset object.\n\n8 | P a g e\n\n>>> frozenset((3,2,4))\n\nfrozenset({2, 3, 4})\n\nRead Python Sets and Booleans for more on frozenset.\n\n24. getattr()\n\ngetattr() returns the value of an object’s attribute.\n\n>>> getattr(orange,\'size\')\n\n7\n\n25. globals()\n\nThis Python built-in functions, returns a dictionary of the current global symbol table.\n\n>>> globals()\n\n{‘__name__’: ‘__main__’, ‘__doc__’: None, ‘__package__’: None, ‘__loader__’: <class ‘_frozen_importlib.BuiltinImporter’>, ‘__spec__’: None, ‘__annotations__’: {}, ‘__builtins__’: <module ‘builtins’ (built-in)>, ‘fruit’: <class ‘__main__.fruit’>, ‘orange’: <__main__.fruit object at 0x05F937D0>, ‘a’: 2, ‘numbers’: [1, 2, 3], ‘i’: (2, 3), ‘x’: 7, ‘b’: 3}\n\n26. hasattr()\n\nLike delattr() and getattr(), hasattr() Python built-in functions, returns True if the object has that attribute.\n\n>>> hasattr(orange,\'si

In [83]:
embeddings = HuggingFaceEmbeddings()

C:\Users\HP\AppData\Local\Temp\ipykernel_17360\3655315981.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [84]:
persist_directory = "vector_db"

In [85]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [86]:
# retriever
retriever = vectordb.as_retriever()

In [87]:
# llm from groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [88]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [89]:
# invoke the qa chain and get a response for user query
query = "what are the function from this pdf "
response = qa_chain.invoke({"query": query})

In [90]:
print(response)

{'query': 'what are the function from this pdf ', 'result': 'Based on the provided context, the following are the Python built-in functions mentioned:\n\n1. divmod()\n2. enumerate()\n3. eval()\n4. exec()\n5. filter()\n6. float()\n7. format()\n8. frozenset()\n9. getattr()\n10. globals()\n\nThese functions are listed with examples and explanations in the provided context. Let me know if you have any specific questions about any of these functions.', 'source_documents': [Document(id='fcb19f81-a174-45ce-95f8-70aa25e2bc9e', metadata={'source': 'python_inbuildfunction.pdf'}, page_content='>>> divmod(3,7)\n\n6 | P a g e\n\n(0, 3)\n\n>>> divmod(7,3)\n\n(2, 1) If you encounter any doubt in Python Built-in Function, Please Comment.\n\n17. enumerate()\n\nThis Python Built In function returns an enumerate object. In other words, it adds a counter to the iterable.\n\n>>> for i in enumerate([\'a\',\'b\',\'c\']):\n\nprint(i)\n\n(0, ‘a’) (1, ‘b’) (2, ‘c’)\n\n18. eval()\n\nThis Function takes a string 

In [91]:
print(response["result"])

Based on the provided context, the following are the Python built-in functions mentioned:

1. divmod()
2. enumerate()
3. eval()
4. exec()
5. filter()
6. float()
7. format()
8. frozenset()
9. getattr()
10. globals()

These functions are listed with examples and explanations in the provided context. Let me know if you have any specific questions about any of these functions.


In [92]:
# invoke the qa chain and get a response for user query
query = "Give me summary of all function from this pdf?"
response = qa_chain.invoke({"query": query})
print(response["result"])
print("*"*30)
print("Source Document:", response["source_documents"][0].metadata["source"])

Here is a summary of the Python built-in functions mentioned in the provided context:

1. **divmod()**: Returns a tuple containing the quotient and remainder when the first argument is divided by the second.
2. **enumerate()**: Returns an enumerate object, which adds a counter to an iterable.
3. **eval()**: Evaluates a string as a Python expression and returns the result.
4. **exec()**: Executes a string as Python code.
5. **filter()**: Filters out items from an iterable for which a condition is True.
6. **float()**: Converts an integer or compatible value to a floating-point number.
7. **format()**: Formats a string using the provided arguments.
8. **frozenset()**: Returns an immutable frozenset object.
9. **getattr()**: Returns the value of an object's attribute.
10. **globals()**: Returns a dictionary of the current global symbol table.

These functions can be used in various contexts, such as:

* **Mathematical operations**: divmod, eval
* **Data manipulation**: enumerate, filter, 

In [93]:
query = "Explain lambda function in Python"
result = qa_chain.invoke({"query": query})
print(result["result"])

**Lambda Functions in Python**

A lambda function in Python is a small, anonymous function that can be defined inline within a larger expression. It is a shorthand way to create small, one-time use functions.

**Syntax**
---------

The syntax for a lambda function is as follows:
```python
lambda arguments: expression
```
Here, `arguments` is a comma-separated list of variables that will be passed to the function, and `expression` is the code that will be executed when the function is called.

**Example**
---------

Here is an example of a simple lambda function:
```python
add = lambda x, y: x + y
print(add(3, 4))  # Output: 7
```
In this example, the lambda function takes two arguments, `x` and `y`, and returns their sum.

**Use Cases**
-------------

Lambda functions are often used in situations where a small, one-time use function is needed. Some common use cases include:

* **Map, Filter, and Reduce**: Lambda functions are often used with the `map()`, `filter()`, and `reduce()` func

In [94]:
query = "Explain math function in Python"

result = qa_chain.invoke({"query": query})

print(result["result"])

The math module in Python provides access to mathematical functions. Here are some of the most commonly used math functions in Python:

1. **Trigonometric Functions:**
   - `math.sin(x)`: Returns the sine of x in radians.
   - `math.cos(x)`: Returns the cosine of x in radians.
   - `math.tan(x)`: Returns the tangent of x in radians.
   - `math.asin(x)`: Returns the angle in radians whose sine is x.
   - `math.acos(x)`: Returns the angle in radians whose cosine is x.
   - `math.atan(x)`: Returns the angle in radians whose tangent is x.

2. **Hyperbolic Functions:**
   - `math.sinh(x)`: Returns the hyperbolic sine of x.
   - `math.cosh(x)`: Returns the hyperbolic cosine of x.
   - `math.tanh(x)`: Returns the hyperbolic tangent of x.

3. **Power and Logarithmic Functions:**
   - `math.pow(x, y)`: Returns x to the power of y.
   - `math.exp(x)`: Returns the value of e to the power of x.
   - `math.log(x)`: Returns the natural logarithm of x.
   - `math.log10(x)`: Returns the base-10 logari